# Frozen BERT: Training-Label Noise Experiments

**MIA5150 — Frozen BERT with a trained classification head**  
**Team:** Callum Anderson and Zaynab Othman Taha

BERT encoder weights will stay frozen; only the classification head will be trained.

Shared data exploration and results analysis are in [data_and_results.ipynb](data_and_results.ipynb).

## Imports


In [17]:
import hashlib
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, set_seed

from IPython.display import display


# 1. Verify GPU, as it will be needed for this experiment

In [18]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


# 2. Load the prepared data and set the experiment conditions

The conditions are **0%, 10%, 20%, and 40% added training-label noise**, with
**seeds 42, 43, and 44**. 

In [19]:
LABEL_COLUMN = "label (target)"
NUMBER_OF_CLASSES = 11
CLASS_IDS = np.arange(NUMBER_OF_CLASSES)
NOISE_PERCENTAGES = (0, 10, 20, 40)
RANDOM_SEEDS = (42, 43, 44)
SPLIT_NAMES = ("train", "validation", "test")

DATA_DIRECTORY = Path("/content")
source_paths = {split: DATA_DIRECTORY / f"{split}_merged.csv" for split in SPLIT_NAMES}
missing_files = [path.name for path in source_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing files in /content: {', '.join(missing_files)}. "
        "Upload the three cleaned CSVs directly to the Colab Files panel."
    )

source_hashes = {
    split: hashlib.sha256(path.read_bytes()).hexdigest()
    for split, path in source_paths.items()
}
data_splits = {
    split: pd.read_csv(path, dtype={"paper_id": "string"})
    for split, path in source_paths.items()
}
for split, frame in data_splits.items():
    assert frame["paper_id"].notna().all() and frame["paper_id"].is_unique, split
    assert pd.api.types.is_integer_dtype(frame[LABEL_COLUMN]), split
    assert set(frame[LABEL_COLUMN]) == set(CLASS_IDS), split
    for text_column in ("title", "abstract"):
        assert frame[text_column].fillna("").str.strip().ne("").all(), split

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert set(data_splits[left]["paper_id"]).isdisjoint(data_splits[right]["paper_id"])

training_data = data_splits["train"]
validation_data = data_splits["validation"]
test_data = data_splits["test"]
original_training_labels = training_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)
original_training_labels.setflags(write=False)
original_split_labels = {split: frame[LABEL_COLUMN].copy(deep=True) for split, frame in data_splits.items()}
class_names = training_data.groupby(LABEL_COLUMN)["classname"].first().reindex(CLASS_IDS)
for frame in data_splits.values():
    assert frame["classname"].eq(frame[LABEL_COLUMN].map(class_names)).all()

print(f"Loaded {sum(len(frame) for frame in data_splits.values()):,} unique papers from {DATA_DIRECTORY}.")


Loaded 23,335 unique papers from /content.


# 3. Add symmetric training-label noise

For each condition, select records **proportionally from each original training class**
and replace each selected label uniformly with **one of the other ten classes**.
Every selected label therefore changes. Validation and test labels stay unchanged.


In [20]:
def add_symmetric_label_noise(original_labels, noise_percent, seed):
    """Return independent noisy labels and a mask of changed records."""
    labels = np.asarray(original_labels)
    if labels.ndim != 1 or labels.size == 0 or not np.issubdtype(labels.dtype, np.integer):
        raise ValueError("Labels must be a nonempty one-dimensional integer array.")
    if not np.isin(labels, CLASS_IDS).all():
        raise ValueError("Labels must be integers from 0 to 10.")
    if not isinstance(noise_percent, (int, np.integer)) or not 0 <= noise_percent <= 100:
        raise ValueError("noise_percent must be an integer between 0 and 100.")
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError("seed must be a nonnegative integer.")

    noisy_labels = labels.copy()
    changed_mask = np.zeros(len(labels), dtype=bool)
    original_class_counts = np.bincount(labels, minlength=NUMBER_OF_CLASSES)
    quota_numerators = original_class_counts * noise_percent
    flips_per_class = quota_numerators // 100
    target_flips = (len(labels) * noise_percent + 50) // 100
    remaining = int(target_flips - flips_per_class.sum())
    remainder_order = np.argsort(-(quota_numerators % 100), kind="stable")
    flips_per_class[remainder_order[:remaining]] += 1

    rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(noise_percent)]))
    for class_id, number_to_flip in enumerate(flips_per_class):
        if number_to_flip == 0:
            continue
        eligible_positions = np.flatnonzero(labels == class_id)
        chosen_positions = rng.choice(eligible_positions, size=int(number_to_flip), replace=False)
        alternative_labels = CLASS_IDS[CLASS_IDS != class_id]
        noisy_labels[chosen_positions] = rng.choice(alternative_labels, size=int(number_to_flip))
        changed_mask[chosen_positions] = True

    assert changed_mask.sum() == target_flips
    assert np.array_equal(changed_mask, noisy_labels != labels)
    return noisy_labels, changed_mask


## 3.1 Generate the reusable label variants

`training_label_variants[(seed, noise_percent)]` holds the labels aligned with
`training_data`. That same entry is used for frozen BERT, LoRA, and full fine-tuning.



In [21]:
training_label_variants = {}
noise_audits = {}

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        noisy_labels, changed_mask = add_symmetric_label_noise(
            original_training_labels, noise_percent, seed
        )
        noisy_labels.setflags(write=False)
        key = (seed, noise_percent)
        training_label_variants[key] = noisy_labels
        noise_audits[key] = pd.DataFrame({
            "paper_id": training_data["paper_id"].to_numpy(copy=True),
            "original_label": original_training_labels.copy(),
            "noisy_label": noisy_labels.copy(),
            "label_changed": changed_mask.copy(),
        })


## 3.2 Verify the scheme

Check exact flip totals, proportional allocation, valid replacement labels,
reproducibility, and alignment to paper IDs. Also confirm the original labels and
all three source files are unchanged. These checks do not establish that the
original disease labels are semantically correct.

In [22]:
original_class_counts = np.bincount(original_training_labels, minlength=NUMBER_OF_CLASSES)
for (seed, noise_percent), noisy_labels in training_label_variants.items():
    audit = noise_audits[(seed, noise_percent)]
    changed = noisy_labels != original_training_labels
    expected_flips = (len(training_data) * noise_percent + 50) // 100
    assert int(changed.sum()) == expected_flips
    assert np.isin(noisy_labels, CLASS_IDS).all()
    assert not np.shares_memory(noisy_labels, original_training_labels)
    assert np.array_equal(audit["paper_id"], training_data["paper_id"])
    assert np.array_equal(audit["original_label"], original_training_labels)
    assert np.array_equal(audit["noisy_label"], noisy_labels)
    assert np.array_equal(audit["label_changed"], changed)

    actual_per_class = np.bincount(original_training_labels[changed], minlength=NUMBER_OF_CLASSES)
    lower_quota = original_class_counts * noise_percent // 100
    upper_quota = (original_class_counts * noise_percent + 99) // 100
    assert ((actual_per_class >= lower_quota) & (actual_per_class <= upper_quota)).all()
    repeated_labels, repeated_mask = add_symmetric_label_noise(
        original_training_labels, noise_percent, seed
    )
    assert np.array_equal(noisy_labels, repeated_labels)
    assert np.array_equal(changed, repeated_mask)

for split, frame in data_splits.items():
    pd.testing.assert_series_equal(frame[LABEL_COLUMN], original_split_labels[split])
    assert hashlib.sha256(source_paths[split].read_bytes()).hexdigest() == source_hashes[split]
assert np.array_equal(training_data[LABEL_COLUMN], original_training_labels)
print("Passed: all 12 variants are reproducible, correctly allocated, and aligned.")
print("Original training, validation, test labels and CSV files remain unchanged.")


Passed: all 12 variants are reproducible, correctly allocated, and aligned.
Original training, validation, test labels and CSV files remain unchanged.


# 4. Tokenize titles and abstracts

BERT-base-uncased tokenizer and 256-token limit.


In [23]:
BERT_CHECKPOINT_NAME = "google-bert/bert-base-uncased"
MAXIMUM_TOKEN_LENGTH = 256

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT_NAME, use_fast=True)


def tokenize_papers(paper_data):
    return bert_tokenizer(
        text=paper_data["title"].tolist(),
        text_pair=paper_data["abstract"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAXIMUM_TOKEN_LENGTH,
        return_tensors="pt",
    )


tokenized_splits = {
    split: tokenize_papers(data_splits[split])
    for split in SPLIT_NAMES
}
training_tokenized_inputs = tokenized_splits["train"]
validation_tokenized_inputs = tokenized_splits["validation"]
test_tokenized_inputs = tokenized_splits["test"]

print("Tokenizer:", BERT_CHECKPOINT_NAME)
print("Maximum sequence length:", MAXIMUM_TOKEN_LENGTH)
for split, encoded_inputs in tokenized_splits.items():
    expected_shape = (len(data_splits[split]), MAXIMUM_TOKEN_LENGTH)
    for input_name in ("input_ids", "attention_mask", "token_type_ids"):
        assert tuple(encoded_inputs[input_name].shape) == expected_shape, (split, input_name)
    print(f"{split}: {expected_shape[0]:,} records × {expected_shape[1]} tokens")


Tokenizer: google-bert/bert-base-uncased
Maximum sequence length: 256
train: 16,334 records × 256 tokens
validation: 3,499 records × 256 tokens
test: 3,502 records × 256 tokens


# 5. Get the frozen BERT vectors for each record

Extract the `[CLS]` vectors once for training and validation, then reuse them
for every noise level and seed. BERT stays frozen and in evaluation mode.

In [24]:
BERT_BATCH_SIZE = 16
set_seed(RANDOM_SEEDS[0])

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime, then rerun the notebook before training.")
computation_device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

bert_model = AutoModel.from_pretrained(BERT_CHECKPOINT_NAME).to(computation_device)
bert_model.requires_grad_(False)
bert_model.eval()
trainable_bert_parameters = sum(
    parameter.numel() for parameter in bert_model.parameters() if parameter.requires_grad
)
assert trainable_bert_parameters == 0
print("Trainable BERT parameters:", trainable_bert_parameters)


def synchronize_gpu():
    if computation_device.type == "cuda":
        torch.cuda.synchronize()


def reset_gpu_peak_memory():
    if computation_device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(computation_device)


def peak_gpu_memory_gib():
    if computation_device.type == "cuda":
        return torch.cuda.max_memory_allocated(computation_device) / (1024 ** 3)
    return np.nan


@torch.no_grad()
def extract_paper_vectors(tokenized_inputs, progress_description):
    bert_model.eval()
    paper_vector_batches = []
    for batch_start in tqdm(
        range(0, len(tokenized_inputs["input_ids"]), BERT_BATCH_SIZE),
        desc=progress_description,
    ):
        tokenized_batch = {
            input_name: input_values[batch_start:batch_start + BERT_BATCH_SIZE].to(computation_device)
            for input_name, input_values in tokenized_inputs.items()
        }
        bert_output = bert_model(**tokenized_batch)
        paper_vector_batches.append(bert_output.last_hidden_state[:, 0, :].cpu())
    return torch.cat(paper_vector_batches, dim=0)


paper_vectors_by_split = {}
encoding_seconds = {}
encoding_peak_memory_gib = {}
feature_source_hashes = dict(source_hashes)
for split in ("train", "validation"):
    synchronize_gpu()
    reset_gpu_peak_memory()
    encoding_start_time = time.perf_counter()
    paper_vectors_by_split[split] = extract_paper_vectors(
        tokenized_splits[split], f"Encoding {split} papers"
    )
    synchronize_gpu()
    encoding_seconds[split] = time.perf_counter() - encoding_start_time
    encoding_peak_memory_gib[split] = peak_gpu_memory_gib()
    assert paper_vectors_by_split[split].shape == (
        len(data_splits[split]), bert_model.config.hidden_size
    )
    print(f"{split}: {tuple(paper_vectors_by_split[split].shape)}, {encoding_seconds[split]:.1f} seconds")

training_paper_vectors = paper_vectors_by_split["train"]
validation_paper_vectors = paper_vectors_by_split["validation"]
validation_target_labels = torch.tensor(validation_data[LABEL_COLUMN].to_numpy(), dtype=torch.long)
true_validation_labels = validation_target_labels.numpy()
assert not training_paper_vectors.requires_grad
assert not validation_paper_vectors.requires_grad

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable BERT parameters: 0


Encoding train papers:   0%|          | 0/1021 [00:00<?, ?it/s]

train: (16334, 768), 18.1 seconds


Encoding validation papers:   0%|          | 0/219 [00:00<?, ?it/s]

validation: (3499, 768), 3.9 seconds


# 6. Train the classification heads

Cross-entropy loss, AdamW, learning rate 0.001, weight decay 0.01, and batch size
32 for **10 fixed epochs**. Train a fresh head for each seed and noise level.
Within each seed, use the same initial weights and batch order across noise levels.

Record validation scores each epoch, but use the **final epoch** for results;
there is no early stopping or best-validation checkpoint selection.

In [25]:
NUMBER_OF_EPOCHS = 10
CLASSIFIER_BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01


@torch.no_grad()
def evaluate_frozen_head(classification_head, paper_vectors, true_labels):
    classification_head.eval()
    class_scores = classification_head(paper_vectors.to(computation_device))
    predictions = class_scores.argmax(dim=1).cpu().numpy()
    return {
        "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(
            true_labels, predictions,
            labels=CLASS_IDS, average="macro", zero_division=0,
        ),
        "predictions": predictions,
    }


print(f"Runs: {len(RANDOM_SEEDS) * len(NOISE_PERCENTAGES)}")
print(f"Stopping rule: final epoch ({NUMBER_OF_EPOCHS})")

Runs: 12
Stopping rule: final epoch (10)


In [26]:
frozen_training_run_id = time.time_ns()
frozen_head_states = {}
frozen_run_seconds = {}
frozen_head_training_seconds = {}
frozen_epoch_validation_seconds = {}
frozen_training_peak_memory_gib = {}
frozen_run_metadata = {}
frozen_training_config = {
    "checkpoint": BERT_CHECKPOINT_NAME, "max_length": MAXIMUM_TOKEN_LENGTH,
    "source_hashes": dict(source_hashes), "epochs": NUMBER_OF_EPOCHS,
    "batch_size": CLASSIFIER_BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
}
training_history_rows = []

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        run_key = (seed, noise_percent)
        set_seed(seed)
        synchronize_gpu()
        reset_gpu_peak_memory()
        classification_head = nn.Linear(
            bert_model.config.hidden_size, NUMBER_OF_CLASSES
        ).to(computation_device)
        classifier_optimizer = torch.optim.AdamW(
            classification_head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
        loss_function = nn.CrossEntropyLoss()
        training_target_labels = torch.tensor(
            training_label_variants[run_key], dtype=torch.long
        )
        assert len(training_paper_vectors) == len(training_target_labels)
        training_batches = DataLoader(
            TensorDataset(training_paper_vectors, training_target_labels),
            batch_size=CLASSIFIER_BATCH_SIZE,
            shuffle=True,
            generator=torch.Generator().manual_seed(seed),
        )
        trainable_head_parameters = sum(
            parameter.numel() for parameter in classification_head.parameters()
            if parameter.requires_grad
        )
        print(f"\nSeed {seed}, noise {noise_percent}% — {trainable_head_parameters:,} trainable parameters")
        synchronize_gpu()
        run_start_time = time.perf_counter()
        head_training_seconds = 0.0
        epoch_validation_seconds = 0.0

        for epoch_number in range(1, NUMBER_OF_EPOCHS + 1):
            synchronize_gpu()
            epoch_training_start = time.perf_counter()
            classification_head.train()
            total_training_loss = 0.0
            for paper_vectors_batch, target_labels_batch in training_batches:
                paper_vectors_batch = paper_vectors_batch.to(computation_device)
                target_labels_batch = target_labels_batch.to(computation_device)
                classifier_optimizer.zero_grad(set_to_none=True)
                class_scores = classification_head(paper_vectors_batch)
                batch_loss = loss_function(class_scores, target_labels_batch)
                batch_loss.backward()
                classifier_optimizer.step()
                total_training_loss += batch_loss.item() * len(target_labels_batch)

            synchronize_gpu()
            head_training_seconds += time.perf_counter() - epoch_training_start
            average_training_loss = total_training_loss / len(training_target_labels)
            validation_start = time.perf_counter()
            validation_metrics = evaluate_frozen_head(
                classification_head, validation_paper_vectors, true_validation_labels
            )
            synchronize_gpu()
            epoch_validation_seconds += time.perf_counter() - validation_start
            training_history_rows.append({
                "seed": seed,
                "noise_percent": noise_percent,
                "epoch": epoch_number,
                "training_loss": average_training_loss,
                "validation_accuracy": validation_metrics["accuracy"],
                "validation_macro_f1": validation_metrics["macro_f1"],
            })
            print(
                f"Epoch {epoch_number:2d}/{NUMBER_OF_EPOCHS}: loss = {average_training_loss:.4f}, "
                f"validation macro-F1 = {validation_metrics['macro_f1']:.4f}"
            )

        synchronize_gpu()
        frozen_run_seconds[run_key] = time.perf_counter() - run_start_time
        frozen_head_training_seconds[run_key] = head_training_seconds
        frozen_epoch_validation_seconds[run_key] = epoch_validation_seconds
        frozen_training_peak_memory_gib[run_key] = peak_gpu_memory_gib()
        frozen_run_metadata[run_key] = {
            "final_epoch": epoch_number,
            "noise_label_sha256": hashlib.sha256(training_target_labels.numpy().tobytes()).hexdigest(),
        }
        frozen_head_states[run_key] = {
            name: value.detach().cpu().clone()
            for name, value in classification_head.state_dict().items()
        }
        assert all(not parameter.requires_grad and parameter.grad is None for parameter in bert_model.parameters())
        print(f"Head training + epoch validation: {frozen_run_seconds[run_key]:.1f} seconds")
        del classification_head, classifier_optimizer, class_scores, batch_loss

frozen_training_history = pd.DataFrame(training_history_rows)
assert len(frozen_head_states) == len(RANDOM_SEEDS) * len(NOISE_PERCENTAGES)
print("\nCompleted all runs; final head weights and epoch history are stored in memory.")


Seed 42, noise 0% — 8,459 trainable parameters
Epoch  1/10: loss = 1.6338, validation macro-F1 = 0.4696
Epoch  2/10: loss = 1.1841, validation macro-F1 = 0.5814
Epoch  3/10: loss = 1.0288, validation macro-F1 = 0.6392
Epoch  4/10: loss = 0.9483, validation macro-F1 = 0.6733
Epoch  5/10: loss = 0.9012, validation macro-F1 = 0.6632
Epoch  6/10: loss = 0.8649, validation macro-F1 = 0.6840
Epoch  7/10: loss = 0.8338, validation macro-F1 = 0.7186
Epoch  8/10: loss = 0.8131, validation macro-F1 = 0.7084
Epoch  9/10: loss = 0.7958, validation macro-F1 = 0.7325
Epoch 10/10: loss = 0.7834, validation macro-F1 = 0.7255
Head training + epoch validation: 2.0 seconds

Seed 42, noise 10% — 8,459 trainable parameters
Epoch  1/10: loss = 1.8105, validation macro-F1 = 0.4755
Epoch  2/10: loss = 1.4597, validation macro-F1 = 0.5686
Epoch  3/10: loss = 1.3449, validation macro-F1 = 0.6187
Epoch  4/10: loss = 1.2892, validation macro-F1 = 0.6636
Epoch  5/10: loss = 1.2586, validation macro-F1 = 0.6276
Ep

# 7. Evaluate on validation

Report each run's accuracy, macro-F1, and per-class scores against the unchanged
validation labels. The test set stays reserved for final evaluation.

In [27]:
validation_result_rows = []
frozen_validation_predictions = {}
frozen_classification_reports = {}
frozen_final_validation_seconds = {}
frozen_validation_peak_memory_gib = {}

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        run_key = (seed, noise_percent)
        synchronize_gpu()
        reset_gpu_peak_memory()
        classification_head = nn.Linear(
            bert_model.config.hidden_size, NUMBER_OF_CLASSES
        ).to(computation_device)
        classification_head.load_state_dict(frozen_head_states[run_key])
        synchronize_gpu()
        validation_start = time.perf_counter()
        validation_metrics = evaluate_frozen_head(
            classification_head, validation_paper_vectors, true_validation_labels
        )
        synchronize_gpu()
        frozen_final_validation_seconds[run_key] = time.perf_counter() - validation_start
        frozen_validation_peak_memory_gib[run_key] = peak_gpu_memory_gib()
        frozen_validation_predictions[run_key] = validation_metrics["predictions"]
        frozen_classification_reports[run_key] = classification_report(
            true_validation_labels, validation_metrics["predictions"],
            labels=CLASS_IDS, target_names=class_names.tolist(),
            output_dict=True, zero_division=0,
        )
        validation_result_rows.append({
            "seed": seed,
            "noise_percent": noise_percent,
            "final_epoch": NUMBER_OF_EPOCHS,
            "accuracy": validation_metrics["accuracy"],
            "macro_f1": validation_metrics["macro_f1"],
            "run_seconds": frozen_run_seconds[run_key],
        })
        del classification_head

frozen_results = pd.DataFrame(validation_result_rows)
print("Final-epoch validation results (accuracy and macro-F1 on a 0–1 scale):")
display(frozen_results.round(4))
frozen_validation_training_run_id = frozen_training_run_id

Final-epoch validation results (accuracy and macro-F1 on a 0–1 scale):


,seed,noise_percent,final_epoch,accuracy,macro_f1,run_seconds
0,42,0,10,0.7522,0.7255,1.9707
1,42,10,10,0.7451,0.7087,1.9677
2,42,20,10,0.7376,0.7091,1.9642
3,42,40,10,0.7279,0.6860,1.9710
4,43,0,10,0.7516,0.7260,2.1706
5,43,10,10,0.7471,0.7227,1.9674
6,43,20,10,0.7185,0.6847,1.9550
7,43,40,10,0.6759,0.5962,1.9335
8,44,0,10,0.7474,0.7142,1.9572
9,44,10,10,0.7385,0.6915,1.9655


In [28]:
REPORT_SEED = 42
REPORT_NOISE_PERCENT = 0
report_key = (REPORT_SEED, REPORT_NOISE_PERCENT)
print(f"Per-class validation scores: seed {REPORT_SEED}, noise {REPORT_NOISE_PERCENT}%")
per_class_validation_report = pd.DataFrame(
    frozen_classification_reports[report_key]
).T.loc[class_names.tolist(), ["precision", "recall", "f1-score", "support"]]
per_class_validation_report["support"] = per_class_validation_report["support"].astype(int)
display(per_class_validation_report.round(4))

Per-class validation scores: seed 42, noise 0%


,precision,recall,f1-score,support
Cardiovascular diseases,0.6734,0.8064,0.7340,780
Chronic kidney disease,0.8551,0.5413,0.6629,109
Chronic respiratory diseases,0.8035,0.7160,0.7572,257
Diabetes mellitus,0.8980,0.7554,0.8205,233
Digestive diseases,0.5690,0.2920,0.3860,113
HIV/AIDS,0.8611,0.8158,0.8378,152
Hepatitis A/B/C/E,0.9020,0.7541,0.8214,61
Mental disorders,0.8333,0.7438,0.7860,363
Musculoskeletal disorders,0.6479,0.7684,0.7030,285
Neoplasms (cancer),0.8230,0.8331,0.8280,815


# 8. Final evaluation on the held-out test set

Report each run's accuracy, macro-F1, per-class precision/recall/F1 and support,
head training time, and peak **PyTorch-allocated GPU memory in GiB** against held-out test set. 

In [29]:
expected_run_keys = {(seed, noise) for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES}
required_measurements = (
    "frozen_run_metadata", "frozen_training_config", "frozen_head_training_seconds",
    "frozen_epoch_validation_seconds", "frozen_training_peak_memory_gib",
    "frozen_final_validation_seconds", "frozen_validation_peak_memory_gib",
    "encoding_peak_memory_gib", "feature_source_hashes",
    "frozen_training_run_id", "frozen_validation_training_run_id",
)
if any(name not in globals() for name in required_measurements):
    raise RuntimeError("Rerun sections 5–7 to record the final heads, timings, and GPU memory.")
assert set(frozen_head_states) == expected_run_keys
assert frozen_validation_training_run_id == frozen_training_run_id, "Rerun section 7 for the current trained heads."
validation_by_run = frozen_results.set_index(["seed", "noise_percent"])
assert validation_by_run.index.is_unique and set(validation_by_run.index) == expected_run_keys
assert validation_by_run["final_epoch"].eq(NUMBER_OF_EPOCHS).all()
for measurements in (frozen_run_metadata, frozen_head_training_seconds, frozen_run_seconds,
                     frozen_epoch_validation_seconds, frozen_training_peak_memory_gib,
                     frozen_final_validation_seconds, frozen_validation_peak_memory_gib):
    assert set(measurements) == expected_run_keys, "Rerun sections 6–7 to finish every run."
assert frozen_training_config == {
    "checkpoint": BERT_CHECKPOINT_NAME, "max_length": MAXIMUM_TOKEN_LENGTH,
    "source_hashes": dict(source_hashes), "epochs": NUMBER_OF_EPOCHS,
    "batch_size": CLASSIFIER_BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
}, "Training settings changed; rerun sections 5–7 before evaluating."
assert feature_source_hashes == source_hashes
for split in SPLIT_NAMES:
    assert hashlib.sha256(source_paths[split].read_bytes()).hexdigest() == source_hashes[split]
    assert data_splits[split][LABEL_COLUMN].equals(original_split_labels[split])
for key in expected_run_keys:
    assert frozen_run_metadata[key]["final_epoch"] == NUMBER_OF_EPOCHS
    assert frozen_run_metadata[key]["noise_label_sha256"] == hashlib.sha256(
        training_label_variants[key].astype(np.int64).tobytes()
    ).hexdigest()

RESULTS_PATH = DATA_DIRECTORY / "run_results.csv"
if RESULTS_PATH.is_file():
    run_registry = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
else:
    run_registry = pd.DataFrame([
        {
            "run_id": f"{'lora_r8' if model == 'lora' else model}_seed{seed}_noise{noise:02d}",
            "model": model, "lora_rank": "8" if model == "lora" else "",
            "seed": str(seed), "requested_noise_percent": str(noise), "status": "planned",
        }
        for model in ("frozen", "lora", "full_finetuning")
        for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES
    ])
assert {"run_id", "model"}.issubset(run_registry.columns)
assert run_registry["run_id"].ne("").all() and run_registry["run_id"].is_unique
run_registry = run_registry.set_index("run_id")
for seed, noise in expected_run_keys:
    run_id = f"frozen_seed{seed}_noise{noise:02d}"
    if run_id in run_registry.index:
        assert run_registry.at[run_id, "model"] == "frozen", run_id

if "test" not in paper_vectors_by_split:
    synchronize_gpu()
    reset_gpu_peak_memory()
    test_encoding_start = time.perf_counter()
    paper_vectors_by_split["test"] = extract_paper_vectors(test_tokenized_inputs, "Encoding test papers")
    synchronize_gpu()
    encoding_seconds["test"] = time.perf_counter() - test_encoding_start
    encoding_peak_memory_gib["test"] = peak_gpu_memory_gib()
test_paper_vectors = paper_vectors_by_split["test"]
assert test_paper_vectors.shape == (len(test_data), bert_model.config.hidden_size)
true_test_labels = test_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)
number_of_runs = len(expected_run_keys)
test_result_rows, per_class_rows = [], []
frozen_test_predictions = {}

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        key = (seed, noise_percent)
        synchronize_gpu()
        reset_gpu_peak_memory()
        classification_head = nn.Linear(bert_model.config.hidden_size, NUMBER_OF_CLASSES).to(computation_device)
        classification_head.load_state_dict(frozen_head_states[key])
        synchronize_gpu()
        inference_start = time.perf_counter()
        test_metrics = evaluate_frozen_head(classification_head, test_paper_vectors, true_test_labels)
        synchronize_gpu()
        test_inference_seconds = time.perf_counter() - inference_start
        test_peak_memory_gib = peak_gpu_memory_gib()
        frozen_test_predictions[key] = test_metrics["predictions"]
        report = classification_report(
            true_test_labels, test_metrics["predictions"],
            labels=CLASS_IDS, output_dict=True, zero_division=0,
        )
        class_scores = [
            {
                "class_id": int(class_id), "class_name": str(class_names.loc[class_id]),
                "precision": report[str(class_id)]["precision"],
                "recall": report[str(class_id)]["recall"],
                "f1": report[str(class_id)]["f1-score"],
                "support": int(report[str(class_id)]["support"]),
            }
            for class_id in CLASS_IDS
        ]
        per_class_rows.extend({"seed": seed, "noise_percent": noise_percent, **row} for row in class_scores)
        validation_row = validation_by_run.loc[key]
        peak_memory = max(
            *encoding_peak_memory_gib.values(), frozen_training_peak_memory_gib[key],
            frozen_validation_peak_memory_gib[key], test_peak_memory_gib,
        )
        test_result_rows.append({
            "run_id": f"frozen_seed{seed}_noise{noise_percent:02d}",
            "model": "frozen", "lora_rank": "", "seed": seed,
            "requested_noise_percent": noise_percent,
            "actual_noise_percent": 100 * np.mean(training_label_variants[key] != original_training_labels),
            "status": "completed", "final_epoch": frozen_run_metadata[key]["final_epoch"],
            "validation_accuracy": validation_row["accuracy"],
            "validation_macro_f1": validation_row["macro_f1"],
            "test_accuracy": test_metrics["accuracy"], "test_macro_f1": test_metrics["macro_f1"],
            "head_training_seconds": frozen_head_training_seconds[key],
            "training_seconds": frozen_head_training_seconds[key] + encoding_seconds["train"] / number_of_runs,
            "training_and_validation_seconds": frozen_run_seconds[key],
            "epoch_validation_seconds": frozen_epoch_validation_seconds[key],
            "final_validation_seconds": frozen_final_validation_seconds[key],
            "test_head_inference_seconds": test_inference_seconds,
            **{f"amortized_{split}_encoding_seconds": encoding_seconds[split] / number_of_runs for split in SPLIT_NAMES},
            "peak_pytorch_allocated_gib": peak_memory,
            "gpu_model": torch.cuda.get_device_name(computation_device) if computation_device.type == "cuda" else "CPU",
            "trainable_parameters": sum(parameter.numel() for parameter in classification_head.parameters()),
            "bert_checkpoint": BERT_CHECKPOINT_NAME, "max_token_length": MAXIMUM_TOKEN_LENGTH,
            **{f"{split}_csv_sha256": source_hashes[split] for split in SPLIT_NAMES},
            "test_per_class_metrics_json": json.dumps(class_scores, ensure_ascii=False, allow_nan=False),
        })
        del classification_head

frozen_final_results = pd.DataFrame(test_result_rows)
assert all(parameter.grad is None and not parameter.requires_grad for parameter in bert_model.parameters())

for column in frozen_final_results.columns:
    if column != "run_id" and column not in run_registry.columns:
        run_registry[column] = ""
previous_comparison_columns = (
    "validation_accuracy_drop_pp", "validation_macro_f1_drop_pp",
    "test_accuracy_drop_pp", "test_macro_f1_drop_pp",
)
for row in frozen_final_results.to_dict("records"):
    run_id = row.pop("run_id")
    if run_id not in run_registry.index:
        run_registry.loc[run_id] = ""
    for column in previous_comparison_columns:
        if column in run_registry.columns:
            run_registry.at[run_id, column] = ""
    for column, value in row.items():
        run_registry.at[run_id, column] = "" if pd.isna(value) else str(value)
temporary_results_path = RESULTS_PATH.with_suffix(".csv.tmp")
run_registry.reset_index().to_csv(temporary_results_path, index=False)
temporary_results_path.replace(RESULTS_PATH)
print(f"Saved all {number_of_runs} frozen runs to {RESULTS_PATH}. Accuracy, macro-F1, precision, and recall use a 0–1 scale.")

print("\nFinal test results for each run:")
display(frozen_final_results[[
    "seed", "requested_noise_percent", "test_accuracy", "test_macro_f1",
    "head_training_seconds", "peak_pytorch_allocated_gib",
]].round(4))

frozen_test_per_class = pd.DataFrame(per_class_rows)
for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        print(f"\nPer-class test scores: seed {seed}, noise {noise_percent}%")
        run_class_scores = frozen_test_per_class.loc[
            frozen_test_per_class["seed"].eq(seed)
            & frozen_test_per_class["noise_percent"].eq(noise_percent),
            ["class_id", "class_name", "precision", "recall", "f1", "support"],
        ]
        display(run_class_scores.reset_index(drop=True).round(4))

print("\nShared BERT feature extraction, measured once per split:")
display(pd.DataFrame({
    "split": SPLIT_NAMES,
    "encoding_seconds": [encoding_seconds[split] for split in SPLIT_NAMES],
    "peak_pytorch_allocated_gib": [encoding_peak_memory_gib[split] for split in SPLIT_NAMES],
}).round(4))

Encoding test papers:   0%|          | 0/219 [00:00<?, ?it/s]

Saved all 12 frozen runs to /content/run_results.csv. Accuracy, macro-F1, precision, and recall use a 0–1 scale.

Final test results for each run:


,seed,requested_noise_percent,test_accuracy,test_macro_f1,head_training_seconds,peak_pytorch_allocated_gib
0,42,0,0.7521,0.7270,1.9514,0.5719
1,42,10,0.7407,0.7052,1.9492,0.5719
2,42,20,0.7319,0.7087,1.9456,0.5719
3,42,40,0.7170,0.6716,1.9521,0.5719
4,43,0,0.7501,0.7268,2.1518,0.5719
5,43,10,0.7450,0.7184,1.9492,0.5719
6,43,20,0.7273,0.6905,1.9365,0.5719
7,43,40,0.6713,0.5994,1.9154,0.5719
8,44,0,0.7416,0.7144,1.9391,0.5719
9,44,10,0.7327,0.6895,1.9472,0.5719



Per-class test scores: seed 42, noise 0%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.6579,0.8028,0.7232,781
1,1,Chronic kidney disease,0.8966,0.4771,0.6228,109
2,2,Chronic respiratory diseases,0.7949,0.7237,0.7576,257
3,3,Diabetes mellitus,0.8586,0.7039,0.7736,233
4,4,Digestive diseases,0.5156,0.3000,0.3793,110
5,5,HIV/AIDS,0.9044,0.8092,0.8542,152
6,6,Hepatitis A/B/C/E,0.9608,0.8167,0.8829,60
7,7,Mental disorders,0.8405,0.7507,0.7931,365
8,8,Musculoskeletal disorders,0.6447,0.7840,0.7075,287
9,9,Neoplasms (cancer),0.8251,0.8402,0.8326,820


...

,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.7788,0.6492,0.7081,781
1,1,Chronic kidney disease,0.8814,0.4771,0.6190,109
2,2,Chronic respiratory diseases,0.6492,0.7704,0.7046,257
3,3,Diabetes mellitus,0.6295,0.8240,0.7138,233
4,4,Digestive diseases,0.5918,0.2636,0.3648,110
5,5,HIV/AIDS,0.7056,0.9145,0.7966,152
6,6,Hepatitis A/B/C/E,0.9200,0.7667,0.8364,60
7,7,Mental disorders,0.6690,0.7973,0.7275,365
8,8,Musculoskeletal disorders,0.8100,0.6237,0.7047,287
9,9,Neoplasms (cancer),0.7702,0.8500,0.8081,820



Per-class test scores: seed 44, noise 40%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8003,0.6005,0.6862,781
1,1,Chronic kidney disease,0.7833,0.4312,0.5562,109
2,2,Chronic respiratory diseases,0.5706,0.7860,0.6612,257
3,3,Diabetes mellitus,0.5256,0.8369,0.6457,233
4,4,Digestive diseases,0.5714,0.1455,0.2319,110
5,5,HIV/AIDS,0.9113,0.7434,0.8188,152
6,6,Hepatitis A/B/C/E,0.9348,0.7167,0.8113,60
7,7,Mental disorders,0.6607,0.8055,0.7259,365
8,8,Musculoskeletal disorders,0.7879,0.6341,0.7027,287
9,9,Neoplasms (cancer),0.7598,0.8524,0.8034,820



Shared BERT feature extraction, measured once per split:


,split,encoding_seconds,peak_pytorch_allocated_gib
0,train,18.1183,0.5719
1,validation,3.8900,0.5719
2,test,3.8856,0.5719
